In [ ]:
import pandas as pd
import joblib
import os

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier


from scipy.stats import randint, uniform

In [ ]:
# load vectors
train_vectors_df = pd.read_csv("./../data/train_pca.csv")
val_vectors_df = pd.read_csv("./../data/val_pca.csv")
test_vectors_df = pd.read_csv("./../data/test_pca.csv")

# load features with labels
train_features_df = pd.read_csv("./../data/train_features.csv")
val_features_df = pd.read_csv("./../data/val_features.csv")
test_features_df = pd.read_csv("./../data/test_features.csv")

In [ ]:
X_train = train_vectors_df
X_train[train_vectors_df.columns] = train_vectors_df[train_vectors_df.columns].astype(float)
y_train = train_features_df['label'].values

X_val = val_vectors_df
X_val[val_vectors_df.columns] = val_vectors_df[val_vectors_df.columns].astype(float)
y_val = val_features_df['label'].values

X_test = test_vectors_df.values
X_test[test_vectors_df.columns] = test_vectors_df[test_vectors_df.columns].astype(float)
y_test = test_features_df['label'].values

In [ ]:
# final shapes
print("Train X shape:", X_train.shape)
print("Train y shape:", y_train.shape)
print("Validation X shape:", X_val.shape)
print("Validation y shape:", y_val.shape)
print("Test X shape:", X_test.shape)
print("Test y shape:", y_test.shape)

# evaluation function 

In [ ]:
def evaluate(model, X_test, y_test, name="model"):
    preds = model.predict(X_test)
    print(f"\n===== {name} =====")
    print("Accuracy:", accuracy_score(y_test, preds))
    print(classification_report(y_test, preds))

# modeling

In [ ]:
svm = SVC()

svm_param_dist = {
    "C": uniform(0.1, 10),
    "gamma": uniform(0.001, 1),
    "kernel": ["rbf", "linear"]
}

svm_search = RandomizedSearchCV(
    svm,
    svm_param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

svm_search.fit(X_train, y_train)

print("Best SVM params:", svm_search.best_params_)

In [ ]:
evaluate(svm_search.best_estimator_, X_val, y_val, "SVM")

In [ ]:

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)

xgb_param_dist = {
    "n_estimators": randint(100, 800),
    "max_depth": randint(3, 12),
    "learning_rate": uniform(0.01, 0.3),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10)
}

xgb_search = RandomizedSearchCV(
    xgb,
    xgb_param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

xgb_search.fit(X_train, y_train)

print("Best XGB params:", xgb_search.best_params_)

In [ ]:
evaluate(xgb_search.best_estimator_, X_val, y_val, "XGBoost")

In [ ]:
rf = RandomForestClassifier(random_state=42)

rf_param_dist = {
    "n_estimators": randint(100, 1000),
    "max_depth": randint(3, 30),
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", None]
}

rf_search = RandomizedSearchCV(
    rf,
    rf_param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

rf_search.fit(X_train, y_train)

print("Best RF params:", rf_search.best_params_)

In [ ]:
evaluate(rf_search.best_estimator_, X_val, y_val, "random forest")

# testing

In [ ]:
evaluate(xgb_search.best_estimator_, X_test, y_test, "XGBoost")
evaluate(svm_search.best_estimator_, X_test, y_test, "SVM")
evaluate(rf_search.best_estimator_, X_test, y_test, "Random Forest")

# save models

In [ ]:
os.makedirs("./../data/final_models", exist_ok=True)
joblib.dump(xgb_search.best_estimator_,"./../data/final_models/xgb_model.pkl")
joblib.dump(svm_search.best_estimator_,"./../data/final_models/svm_model.pkl")
joblib.dump(rf_search.best_estimator_,"./../data/final_models/rf_model.pkl")
print("Models saved successfully.")